In [6]:
pip install -q transformers accelerate bitsandbytes nnsight

In [7]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from nnsight import LanguageModel

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = LanguageModel("Qwen/Qwen2.5-1.5B-Instruct", device_map="auto", quantization_config=quant_config)


/tmp/ipykernel_505/3996131207.py:13: NNsightDeprecationWarning: LanguageModel is deprecated; use TransformersModel(repo_id, task='text-generation') instead.
  model = LanguageModel("Qwen/Qwen2.5-1.5B-Instruct", device_map="auto", quantization_config=quant_config)


In [8]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from nnsight import TransformersModel

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# Updated to use TransformersModel and explicitly set the task
model = TransformersModel(
    "Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    device_map="auto",
    quantization_config=quant_config
)

In [9]:
# Format prompts using the Qwen chat template
clean_text = "Jim has 5 apples. He gives 2 to his friend. How many apples does he have left?"
noop_text = "Jim has 5 apples. He gives 2 to his friend. The friend has a red car. How many apples does he have left?"

clean_prompt = tokenizer.apply_chat_template([{"role": "user", "content": clean_text}], tokenize=False, add_generation_prompt=True)
noop_prompt = tokenizer.apply_chat_template([{"role": "user", "content": noop_text}], tokenize=False, add_generation_prompt=True)

target_layer = 14 # Mid-layer where attention leakage typically peaks

# Trace the clean prompt and save the final token's residual stream
with model.trace(clean_prompt):
    clean_hidden = model.model.layers[target_layer].output[0, -1, :].save()

# Trace the NoOp prompt and save the final token's residual stream
with model.trace(noop_prompt):
    noop_hidden = model.model.layers[target_layer].output[0, -1, :].save()

# Calculate the Difference-in-Means vector and normalize it directly on the tensors
v_steer = noop_hidden - clean_hidden
v_norm = v_steer / torch.norm(v_steer)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
import torch
from tqdm import tqdm

# Qwen2.5-1.5B has 28 layers (indexed 0 to 27)
num_layers = 28
layer_results = {}

for target_layer in tqdm(range(num_layers), desc="Scanning Layers"):
    # 1. Extract the Difference-in-Means vector for the current layer
    with model.trace(clean_prompt):
        clean_hidden = model.model.layers[target_layer].output[0, -1, :].save()

    with model.trace(noop_prompt):
        noop_hidden = model.model.layers[target_layer].output[0, -1, :].save()

    v_steer = noop_hidden - clean_hidden
    v_norm = v_steer / torch.norm(v_steer)

    # 2. Generate the answer with surgical ablation active
    with model.generate(noop_prompt, max_new_tokens=50, do_sample=False) as generator:

        # Grab the live residual stream tensor
        hidden_states = model.model.layers[target_layer].output

        # Calculate projection and ablate
        projection = (hidden_states * v_norm).sum(dim=-1, keepdim=True) * v_norm
        model.model.layers[target_layer].output = hidden_states - projection

        # FIX: Save the generated tokens BEFORE the context block exits
        saved_output = generator.result.save()

    # 3. Decode the output and store it AFTER the block has executed
    # We slice [len(noop_prompt_tokens):] if you only want the new tokens,
    # or just decode the whole thing.
    steered_output = tokenizer.decode(saved_output[0], skip_special_tokens=True)
    layer_results[target_layer] = steered_output

Scanning Layers: 100%|██████████| 28/28 [01:49<00:00,  3.91s/it]


In [11]:
correct_answer = "3"
print("--- Ablation Results by Layer ---")

for layer, text in layer_results.items():
    if correct_answer in text:
        print(f"Layer {layer}: SUCCESS - Distractor filtered.")
        print(f"Output: {text}\n")
    else:
        print(f"Layer {layer}: FAILED")

--- Ablation Results by Layer ---
Layer 0: SUCCESS - Distractor filtered.
Output: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Jim has 5 apples. He gives 2 to his friend. The friend has a red car. How many apples does he have left?
assistant
Jim originally had 5 apples. After giving 2 to his friend, Jim would have:

\[ 5 - 2 = 3 \]

So, Jim now has 3 apples.

The friend who received the apples from Jim also has a

Layer 1: SUCCESS - Distractor filtered.
Output: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Jim has 5 apples. He gives 2 to his friend. The friend has a red car. How many apples does he have left?
assistant
Jim originally had 5 apples. After giving 2 to his friend, Jim would have:

\[ 5 - 2 = 3 \]

So, Jim has 3 apples left.

The information about the friend's red car is irrelevant to

Layer 2: SUCCESS - Distractor filtered.
Output: system
You are Qwen, created by Alibaba Cloud. You are a helpful assista